# Lab 3: Local Model Training and Parameter Aggregation

This notebook performs one full-batch linear-model update for training groups A and B, then compares equal-weight and sample-count-weighted parameter aggregation.

## 1. Imports and data

The training arrays and shared test set below match the assignment handout. The test set is used only for evaluation.

In [1]:
import numpy as np
import pandas as pd

x_a = np.array([0, 1, 2, 3], dtype=float)
y_a = np.array([1, 3, 5, 7], dtype=float)

x_b = np.array([0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5], dtype=float)
y_b = np.array([2, 3, 4, 5, 6, 7, 8, 9], dtype=float)

x_test = np.array([0.25, 1.25, 2.25, 3.25], dtype=float)
y_test = np.array([2, 4, 6, 8], dtype=float)

n_a = len(x_a)
n_b = len(x_b)
print(f'n_a = {n_a}')
print(f'n_b = {n_b}')

n_a = 4
n_b = 8


## 2. One local full-batch update

For $\hat{y}=wx+b$, initialize $w=0$ and $b=0$. Compute every prediction error before changing either parameter, then apply one update with learning rate $0.05$.

In [2]:
learning_rate = 0.05
initial_w = 0.0
initial_b = 0.0

def one_local_update(x, y, learning_rate=0.05):
    error = initial_w * x + initial_b - y
    grad_w = 2 * np.mean(error * x)
    grad_b = 2 * np.mean(error)
    w_new = initial_w - learning_rate * grad_w
    b_new = initial_b - learning_rate * grad_b
    return grad_w, grad_b, w_new, b_new

grad_w_a, grad_b_a, w_a, b_a = one_local_update(x_a, y_a, learning_rate)
grad_w_b, grad_b_b, w_b, b_b = one_local_update(x_b, y_b, learning_rate)

local_update_table = pd.DataFrame({
    'group': ['A', 'B'],
    'grad_w': [grad_w_a, grad_w_b],
    'grad_b': [grad_b_a, grad_b_b],
    'w': [w_a, w_b],
    'b': [b_a, b_b],
})
display(local_update_table)

,group,grad_w,grad_b,w,b
0,A,-17.0,-8.0,0.850,0.40
1,B,-24.5,-11.0,1.225,0.55


## 3. Aggregate and evaluate

The local parameter pairs are reused for both methods. Equal weighting uses $0.5$ for each group. Sample weighting uses $n_a/(n_a+n_b)$ and $n_b/(n_a+n_b)$.

In [3]:
weight_equal_a = 0.5
weight_equal_b = 0.5
weight_sample_a = n_a / (n_a + n_b)
weight_sample_b = n_b / (n_a + n_b)

def aggregate_and_evaluate(weight_a, weight_b):
    w = weight_a * w_a + weight_b * w_b
    b = weight_a * b_a + weight_b * b_b
    predictions = w * x_test + b
    test_mse = np.mean((predictions - y_test) ** 2)
    return w, b, test_mse

equal_w, equal_b, equal_test_mse = aggregate_and_evaluate(weight_equal_a, weight_equal_b)
sample_w, sample_b, sample_test_mse = aggregate_and_evaluate(weight_sample_a, weight_sample_b)

comparison = pd.DataFrame([
    {
        'method': 'Equal-weight',
        'n_a': n_a, 'n_b': n_b,
        'w_a': w_a, 'b_a': b_a, 'w_b': w_b, 'b_b': b_b,
        'weight_a': weight_equal_a, 'weight_b': weight_equal_b,
        'w': equal_w, 'b': equal_b, 'test_mse': equal_test_mse,
    },
    {
        'method': 'Sample-weighted',
        'n_a': n_a, 'n_b': n_b,
        'w_a': w_a, 'b_a': b_a, 'w_b': w_b, 'b_b': b_b,
        'weight_a': weight_sample_a, 'weight_b': weight_sample_b,
        'w': sample_w, 'b': sample_b, 'test_mse': sample_test_mse,
    },
])
display(comparison)

,method,n_a,n_b,w_a,b_a,w_b,b_b,weight_a,weight_b,w,b,test_mse
0,Equal-weight,4,8,0.85,0.4,1.225,0.55,0.500000,0.500000,1.0375,0.475,8.498721
1,Sample-weighted,4,8,0.85,0.4,1.225,0.55,0.333333,0.666667,1.1000,0.500,7.643125


## 4. Short comparison

Group B has more influence under sample-weighted aggregation because it contains more training examples ($n_b=8$ versus $n_a=4$). Equal weighting gives both local models the same influence regardless of sample count. The sample-weighted aggregation performs better on the shared test set, achieving a lower test MSE compared to equal-weight aggregation.

In [4]:
csv_columns = ['n_a', 'n_b', 'weight_a', 'weight_b', 'w', 'b', 'test_mse']
equal_weight_results = comparison.loc[comparison['method'] == 'Equal-weight', csv_columns]
sample_weighted_results = comparison.loc[comparison['method'] == 'Sample-weighted', csv_columns]

equal_weight_results.to_csv('equal_weight_results.csv', index=False)
sample_weighted_results.to_csv('sample_weighted_results.csv', index=False)

print('Created equal_weight_results.csv')
print('Created sample_weighted_results.csv')
display(equal_weight_results)
display(sample_weighted_results)

Created equal_weight_results.csv
Created sample_weighted_results.csv


,n_a,n_b,weight_a,weight_b,w,b,test_mse
0,4,8,0.5,0.5,1.0375,0.475,8.498721


,n_a,n_b,weight_a,weight_b,w,b,test_mse
1,4,8,0.333333,0.666667,1.1,0.5,7.643125


In [5]:
assert (n_a, n_b) == (4, 8)
assert np.isfinite(comparison[['weight_a', 'weight_b', 'w', 'b', 'test_mse']].to_numpy()).all()
assert (comparison['test_mse'] >= 0).all()
assert list(pd.read_csv('equal_weight_results.csv').columns) == csv_columns
assert list(pd.read_csv('sample_weighted_results.csv').columns) == csv_columns
assert len(pd.read_csv('equal_weight_results.csv')) == 1
assert len(pd.read_csv('sample_weighted_results.csv')) == 1
print('Validation passed: counts, finite values, nonnegative MSE, and CSV schemas are correct.')

Validation passed: counts, finite values, nonnegative MSE, and CSV schemas are correct.
